In [ ]:
Celda 1: Descargar stocks.txt, configurar Spark y cargar el archivo (Paso 1)

In [4]:
import os
from pyspark.sql import SparkSession

# 1. Descargar el dataset stocks.txt dentro de la carpeta DATOS
!mkdir -p DATOS
!wget -q "https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt" -O ./DATOS/stocks.txt

# 2. Crear SparkSession personalizada ("AnalisisStocks" y 512m de memoria)
spark = (SparkSession.builder
    .appName("AnalisisStocks")
    .config("spark.executor.memory", "512m")
    .getOrCreate())

sc = spark.sparkContext

# 3. Construir ruta absoluta con prefijo file:// y cargar archivo
path_relativo = "./DATOS/stocks.txt"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto

rdd = sc.textFile(path_final)

# 4. Verificar carga mostrando las primeras 5 líneas
print("Primeras 5 líneas del archivo:")
for linea in rdd.take(5):
    print(linea)

Primeras 5 líneas del archivo:
id,name,category,quantity,price
1,iPhone,Electronics,10,899.99
2,Macbook,Electronics,5,1299.99
3,iPad,Electronics,15,499.99
4,Samsung TV,Electronics,8,799.99


In [ ]:
Limpieza y Parseo de Datos (Paso 2)

In [5]:
# Paso 2: Limpieza y parseo a (símbolo, fecha, precio, volumen)
def parsear_linea(linea):
    try:
        partes = linea.split(",")
        simbolo = partes[0].strip()
        fecha = partes[1].strip()
        precio = float(partes[2].strip())
        volumen = int(partes[3].strip())
        return (simbolo, fecha, precio, volumen)
    except:
        return None

# Aplicar función y filtrar líneas inválidas o encabezados
rdd_parseado = rdd.map(parsear_linea).filter(lambda x: x is not None)

print(f"Líneas válidas procesadas: {rdd_parseado.count()}")
print("Muestra de registros limpios:", rdd_parseado.take(3))

Líneas válidas procesadas: 0
Muestra de registros limpios: []
